#  Problem 3
## Borges and Daripa JCP Paper

### Problem
$$
\begin{align*}
u(x,y) &= \frac{e^x+e^y}{1+xy}\\
f(x,y) &=\frac{e^x \left(1 - 2 y - 2 x (-1 + y) y + 2 y^2 + x^2 (2 + y^2)\right)+
e^y \left(1 + 2 x (-1 + y) + 2 y^2 + x^2 (2 - 2 y + y^2)\right)}{(1 + x y)^3}
\end{align*}
$$

Table 4, n=64 varying M values, trapezoidal and simpsons, dirichlet and neumann, unit disk

Table 5,n=64 varying M values, trapezoidal and simpsons, dirichlet and neumann, B(0,.5)


# Imports

In [1]:
import numpy as np
import warnings
import matplotlib.pyplot as plt
from matplotlib.tri import Triangulation
import numpy as np
import time
import matplotlib.pyplot as plt
import pandas as pd

import os, sys

# Main project root
repo_root = r"C:\Users\charl\OneDrive\Documents\GitHub\Pyle_Daripa_Research"
os.chdir(repo_root)
if repo_root not in sys.path:
    sys.path.append(repo_root)


from poisson_solver_testing import (
    generate_grid_values,
    generate_uniform_radial,
    generate_nonuniform_radial,
    generate_uniform_azimuthal,
    generate_fixed_nonuniform_azimuthal,
    generate_nonuniform_azimuthal,
    generate_cartesian_grid_on_disk,
    trap_2d_on_disk,
    compute_error_metrics,
    plot_on_disk_with_error,
    plot_on_disk,
    poisson_solver
)




# Problem Setup

In [2]:
# Problem Setup — Problem 3 (Borges–Daripa JCP)

# Radii for Tables 4 and 5
R1 = 1.0    # unit disk B(0,1)
R2 = 0.5    # disk B(0,0.5)

# Radial mesh type: keep radial grid uniform here
rad_unif = 1

# ---------------------------------------------------------------------
# Solution and data functions for Problem 3
#   u(x,y) = (e^x + e^y) / (1 + x y)
#
#   f(x,y) = [ e^x (1 − 2 y − 2 x (-1 + y) y + 2 y^2 + x^2 (2 + y^2))
#            + e^y (1 + 2 x (-1 + y) + 2 y^2 + x^2 (2 − 2 y + y^2)) ]
#            / (1 + x y)^3
# ---------------------------------------------------------------------
u = lambda x, y: (np.exp(x) + np.exp(y)) / (1.0 + x * y)

f = lambda x, y: (
    np.exp(x) * (1.0 - 2.0*y - 2.0*x*(-1.0 + y)*y + 2.0*y**2 + x**2*(2.0 + y**2))
    + np.exp(y) * (1.0 + 2.0*x*(-1.0 + y) + 2.0*y**2 + x**2*(2.0 - 2.0*y + y**2))
) / (1.0 + x * y)**3

# --- Dirichlet and Neumann boundary data for Problem 3 ---

# Dirichlet boundary data: u on the boundary
g_dirichlet = lambda x, y: u(x, y)

# Partial derivatives of u
def u_x(x, y):
    num   = np.exp(x) + np.exp(y)
    den   = 1.0 + x * y
    num_x = np.exp(x)
    den_x = y
    return (num_x * den - num * den_x) / den**2

def u_y(x, y):
    num   = np.exp(x) + np.exp(y)
    den   = 1.0 + x * y
    num_y = np.exp(y)
    den_y = x
    return (num_y * den - num * den_y) / den**2

# Neumann boundary data factory: ∂u/∂n on r = R (n = (x, y)/R on circle of radius R)
def make_g_neumann(R):
    return lambda x, y: (u_x(x, y) * x + u_y(x, y) * y) / R

# N and M values (Problem 3: Tables 4 and 5 use fixed N = 64, varying M)
M_values = [64, 128, 256, 512, 1024, 2048]
N_fixed  = 64

# ----------------------------------------------------------
# Methods to test
# ----------------------------------------------------------
methods = [
    dict(
        name="uniform_fft",
        label="Uniform Mesh",
        azu_unif=2,
        mesh_kind=None,
        use_nudft=None,
    ),
    dict(
        name="Fixed Nonuniform Mesh - NUDFT",
        label="Fixed Nonuniform Mesh - NUDFT",
        azu_unif=1,
        mesh_kind="rand",
        use_nudft=True,
    ),
    dict(
        name="Fixed Nonuniform Mesh - NUFFT",
        label="Fixed Nonuniform Mesh - NUFFT",
        azu_unif=1,
        mesh_kind="rand",
        use_nudft=False,
    ),
]

BC_MAP = {
    "dirichlet": 1,
    "neumann": 2,
}

QUAD_MAP = {
    "trapezoidal": 1,
    "simpson": 2,
}

# Helper Functions

In [3]:
# Helper Functions
ANGLE_MESH_CACHE = {}

def get_cached_angle_mesh(method_cfg, N, M):
    azu_unif    = method_cfg["azu_unif"]
    mesh_kind   = method_cfg["mesh_kind"]
    method_name = method_cfg["name"]

    if azu_unif == 2:
        # Uniform angles: just use the standard uniform mesh of size N
        return generate_uniform_azimuthal(N)

    if azu_unif == 1:
        # Fixed nonuniform mesh, same for all radii for this method and N
        key = (method_name, N, mesh_kind)
        if key not in ANGLE_MESH_CACHE:
            ANGLE_MESH_CACHE[key] = generate_fixed_nonuniform_azimuthal(
                N, kind=mesh_kind or "rand"
            )
        return ANGLE_MESH_CACHE[key]

    if azu_unif == 0:
        # Fully nonuniform: different azimuthal mesh at each radius
        key = (method_name, N, M, mesh_kind)
        if key not in ANGLE_MESH_CACHE:
            ANGLE_MESH_CACHE[key] = generate_nonuniform_azimuthal(
                N, M, kind=mesh_kind or "rand"
            )
        return ANGLE_MESH_CACHE[key]

    raise ValueError("Incorrect index for 'azu_unif'")


def build_radial_mesh(M, R):
    if rad_unif == 1:
        return generate_uniform_radial(M, R)
    return generate_nonuniform_radial(M, R)


def run_single_case(N, M, R, method_cfg, bc_name, quad_name):
    bc_choice = BC_MAP[bc_name]
    quad_rule = QUAD_MAP[quad_name]

    azu_unif = method_cfg["azu_unif"]
    use_nudft = method_cfg["use_nudft"]

    iRadius = build_radial_mesh(M, R)
    iAngle  = get_cached_angle_mesh(method_cfg, N, M)

    x_coord, y_coord = generate_cartesian_grid_on_disk(iAngle, iRadius)

    # interior data and true solution
    f_values = generate_grid_values(f, x_coord, y_coord)
    u_true   = generate_grid_values(u, x_coord, y_coord)

    # boundary data
    if bc_choice == 1:  # Dirichlet
        g_values = generate_grid_values(
            g_dirichlet, x_coord[:, M - 1], y_coord[:, M - 1]
        )
    elif bc_choice == 2:  # Neumann
        # use radius-dependent Neumann data: g_neumann(x,y) = ∂u/∂n on r = R
        g_neumann_R = make_g_neumann(R)
        g_values = generate_grid_values(
            g_neumann_R, x_coord[:, M - 1], y_coord[:, M - 1]
        )
    else:
        raise ValueError("Unknown bc_choice")

    # n = 0 mode for Neumann (phi_0), empty for Dirichlet
    if bc_choice == 2:
        u_fourier_0 = u_true.mean(axis=0)
    else:
        u_fourier_0 = np.array([])

    t0 = time.perf_counter()
    try:
        u_approx = poisson_solver(
            f_values, g_values, u_fourier_0,
            N, M, iRadius, iAngle, R,
            quad_rule, bc_choice,
            rad_unif, azu_unif,
            use_nudft_angular=(use_nudft if use_nudft is not None else False),
            maxiter_nufft=50,
            tol_nufft=1e-8,
        )
        runtime = time.perf_counter() - t0

        _, linf_rel, _, l2_rel = compute_error_metrics(
            u_approx, u_true, iRadius, iAngle
        )

    except MemoryError:
        runtime = np.nan
        linf_rel = np.nan
        l2_rel = np.nan

    return {
        "method": method_cfg["name"],
        "label": method_cfg["label"],
        "N": N,
        "M": M,
        "R": R,
        "bc": bc_name,
        "quad": quad_name,
        "runtime": runtime,
        "L_inf_rel": linf_rel,
        "L2_rel": l2_rel,
    }

# Run Code, table 4 and 5

In [4]:

# Table 4: R = R1 (unit disk)
table4_results = []
for method in methods:
    print(f"\n{method['label']} -- Table 4 (R = {R1})")
    for M in M_values:
        for quad_name in ["trapezoidal", "simpson"]:
            for bc_name in ["dirichlet", "neumann"]:
                res = run_single_case(
                    N=N_fixed,
                    M=M,
                    R=R1,
                    method_cfg=method,
                    bc_name=bc_name,
                    quad_name=quad_name,
                )
                table4_results.append(res)

df_table4 = pd.DataFrame(table4_results)

# Table 5: R = R2 (B(0,0.5))
table5_results = []
for method in methods:
    print(f"\n{method['label']} -- Table 5 (R = {R2})")
    for M in M_values:
        for quad_name in ["trapezoidal", "simpson"]:
            for bc_name in ["dirichlet", "neumann"]:
                res = run_single_case(
                    N=N_fixed,
                    M=M,
                    R=R2,
                    method_cfg=method,
                    bc_name=bc_name,
                    quad_name=quad_name,
                )
                table5_results.append(res)

df_table5 = pd.DataFrame(table5_results)


Uniform Mesh -- Table 4 (R = 1.0)


C:\Users\charl\OneDrive\Documents\GitHub\Pyle_Daripa_Research\poisson_solver_testing\radial.py:453: RuntimeWarning: divide by zero encountered in scalar divide
  ratio = r_m[i] / r_m[i - 2]



Fixed Nonuniform Mesh - NUDFT -- Table 4 (R = 1.0)


C:\Users\charl\OneDrive\Documents\GitHub\Pyle_Daripa_Research\poisson_solver_testing\radial.py:453: RuntimeWarning: divide by zero encountered in scalar divide
  ratio = r_m[i] / r_m[i - 2]
C:\Users\charl\OneDrive\Documents\GitHub\Pyle_Daripa_Research\poisson_solver_testing\radial.py:453: RuntimeWarning: divide by zero encountered in scalar divide
  ratio = r_m[i] / r_m[i - 2]
C:\Users\charl\OneDrive\Documents\GitHub\Pyle_Daripa_Research\poisson_solver_testing\radial.py:453: RuntimeWarning: divide by zero encountered in scalar divide
  ratio = r_m[i] / r_m[i - 2]
C:\Users\charl\OneDrive\Documents\GitHub\Pyle_Daripa_Research\poisson_solver_testing\radial.py:453: RuntimeWarning: divide by zero encountered in scalar divide
  ratio = r_m[i] / r_m[i - 2]
C:\Users\charl\OneDrive\Documents\GitHub\Pyle_Daripa_Research\poisson_solver_testing\radial.py:453: RuntimeWarning: divide by zero encountered in scalar divide
  ratio = r_m[i] / r_m[i - 2]
C:\Users\charl\OneDrive\Documents\GitHub\Pyle_Dari


Fixed Nonuniform Mesh - NUFFT -- Table 4 (R = 1.0)


...

C:\Users\charl\OneDrive\Documents\GitHub\Pyle_Daripa_Research\poisson_solver_testing\radial.py:453: RuntimeWarning: divide by zero encountered in scalar divide
  ratio = r_m[i] / r_m[i - 2]



Fixed Nonuniform Mesh - NUDFT -- Table 5 (R = 0.5)


C:\Users\charl\OneDrive\Documents\GitHub\Pyle_Daripa_Research\poisson_solver_testing\radial.py:453: RuntimeWarning: divide by zero encountered in scalar divide
  ratio = r_m[i] / r_m[i - 2]
C:\Users\charl\OneDrive\Documents\GitHub\Pyle_Daripa_Research\poisson_solver_testing\radial.py:453: RuntimeWarning: divide by zero encountered in scalar divide
  ratio = r_m[i] / r_m[i - 2]
C:\Users\charl\OneDrive\Documents\GitHub\Pyle_Daripa_Research\poisson_solver_testing\radial.py:453: RuntimeWarning: divide by zero encountered in scalar divide
  ratio = r_m[i] / r_m[i - 2]
C:\Users\charl\OneDrive\Documents\GitHub\Pyle_Daripa_Research\poisson_solver_testing\radial.py:453: RuntimeWarning: divide by zero encountered in scalar divide
  ratio = r_m[i] / r_m[i - 2]
C:\Users\charl\OneDrive\Documents\GitHub\Pyle_Daripa_Research\poisson_solver_testing\radial.py:453: RuntimeWarning: divide by zero encountered in scalar divide
  ratio = r_m[i] / r_m[i - 2]
C:\Users\charl\OneDrive\Documents\GitHub\Pyle_Dari


Fixed Nonuniform Mesh - NUFFT -- Table 5 (R = 0.5)


C:\Users\charl\OneDrive\Documents\GitHub\Pyle_Daripa_Research\poisson_solver_testing\radial.py:453: RuntimeWarning: divide by zero encountered in scalar divide
  ratio = r_m[i] / r_m[i - 2]
C:\Users\charl\OneDrive\Documents\GitHub\Pyle_Daripa_Research\poisson_solver_testing\radial.py:453: RuntimeWarning: divide by zero encountered in scalar divide
  ratio = r_m[i] / r_m[i - 2]
C:\Users\charl\OneDrive\Documents\GitHub\Pyle_Daripa_Research\poisson_solver_testing\radial.py:453: RuntimeWarning: divide by zero encountered in scalar divide
  ratio = r_m[i] / r_m[i - 2]
C:\Users\charl\OneDrive\Documents\GitHub\Pyle_Daripa_Research\poisson_solver_testing\radial.py:453: RuntimeWarning: divide by zero encountered in scalar divide
  ratio = r_m[i] / r_m[i - 2]
C:\Users\charl\OneDrive\Documents\GitHub\Pyle_Daripa_Research\poisson_solver_testing\radial.py:453: RuntimeWarning: divide by zero encountered in scalar divide
  ratio = r_m[i] / r_m[i - 2]
C:\Users\charl\OneDrive\Documents\GitHub\Pyle_Dari

# View Results

In [6]:
def dash_if_nan(x):
    return "—" if pd.isna(x) else f"{x:.1e}"

# TABLE 4: Problem 3, R = R1
for method in methods:
    label = method["label"]
    name  = method["name"]

    print(f"\n{'='*80}")
    print(f"{label} : TABLE 4 (Problem 3, R = {R1})")
    print(f"{'='*80}")

    df4 = df_table4[df_table4["method"] == name].copy()

    cols = []

    # Trapezoidal, Dirichlet
    cols.append(("Trapezoidal rule", "Dirichlet", "||·||∞",
                 df4[(df4["quad"] == "trapezoidal") & (df4["bc"] == "dirichlet")]
                    .set_index("M")["L_inf_rel"]))
    cols.append(("Trapezoidal rule", "Dirichlet", "||·||2",
                 df4[(df4["quad"] == "trapezoidal") & (df4["bc"] == "dirichlet")]
                    .set_index("M")["L2_rel"]))

    # Trapezoidal, Neumann
    cols.append(("Trapezoidal rule", "Neumann", "||·||∞",
                 df4[(df4["quad"] == "trapezoidal") & (df4["bc"] == "neumann")]
                    .set_index("M")["L_inf_rel"]))
    cols.append(("Trapezoidal rule", "Neumann", "||·||2",
                 df4[(df4["quad"] == "trapezoidal") & (df4["bc"] == "neumann")]
                    .set_index("M")["L2_rel"]))

    # Simpson, Dirichlet
    cols.append(("Simpson's rule", "Dirichlet", "||·||∞",
                 df4[(df4["quad"] == "simpson") & (df4["bc"] == "dirichlet")]
                    .set_index("M")["L_inf_rel"]))
    cols.append(("Simpson's rule", "Dirichlet", "||·||2",
                 df4[(df4["quad"] == "simpson") & (df4["bc"] == "dirichlet")]
                    .set_index("M")["L2_rel"]))

    # Simpson, Neumann
    cols.append(("Simpson's rule", "Neumann", "||·||∞",
                 df4[(df4["quad"] == "simpson") & (df4["bc"] == "neumann")]
                    .set_index("M")["L_inf_rel"]))
    cols.append(("Simpson's rule", "Neumann", "||·||2",
                 df4[(df4["quad"] == "simpson") & (df4["bc"] == "neumann")]
                    .set_index("M")["L2_rel"]))

    data   = { (r, bc, norm): series for (r, bc, norm, series) in cols }
    pivot4 = pd.DataFrame(data)
    pivot4.index.name = "M"
    pivot4 = pivot4.reindex(M_values)
    pivot4 = pivot4.sort_index(axis=1, level=[0, 1, 2])

    display(pivot4.map(dash_if_nan))

# TABLE 5: Problem 3, R = R2
for method in methods:
    label = method["label"]
    name  = method["name"]

    print(f"\n{'='*80}")
    print(f"{label} : TABLE 5 (Problem 3, R = {R2})")
    print(f"{'='*80}")

    df5 = df_table5[df_table5["method"] == name].copy()

    cols = []

    # Trapezoidal, Dirichlet
    cols.append(("Trapezoidal rule", "Dirichlet", "||·||∞",
                 df5[(df5["quad"] == "trapezoidal") & (df5["bc"] == "dirichlet")]
                    .set_index("M")["L_inf_rel"]))
    cols.append(("Trapezoidal rule", "Dirichlet", "||·||2",
                 df5[(df5["quad"] == "trapezoidal") & (df5["bc"] == "dirichlet")]
                    .set_index("M")["L2_rel"]))

    # Trapezoidal, Neumann
    cols.append(("Trapezoidal rule", "Neumann", "||·||∞",
                 df5[(df5["quad"] == "trapezoidal") & (df5["bc"] == "neumann")]
                    .set_index("M")["L_inf_rel"]))
    cols.append(("Trapezoidal rule", "Neumann", "||·||2",
                 df5[(df5["quad"] == "trapezoidal") & (df5["bc"] == "neumann")]
                    .set_index("M")["L2_rel"]))

    # Simpson, Dirichlet
    cols.append(("Simpson's rule", "Dirichlet", "||·||∞",
                 df5[(df5["quad"] == "simpson") & (df5["bc"] == "dirichlet")]
                    .set_index("M")["L_inf_rel"]))
    cols.append(("Simpson's rule", "Dirichlet", "||·||2",
                 df5[(df5["quad"] == "simpson") & (df5["bc"] == "dirichlet")]
                    .set_index("M")["L2_rel"]))

    # Simpson, Neumann
    cols.append(("Simpson's rule", "Neumann", "||·||∞",
                 df5[(df5["quad"] == "simpson") & (df5["bc"] == "neumann")]
                    .set_index("M")["L_inf_rel"]))
    cols.append(("Simpson's rule", "Neumann", "||·||2",
                 df5[(df5["quad"] == "simpson") & (df5["bc"] == "neumann")]
                    .set_index("M")["L2_rel"]))

    data   = { (r, bc, norm): series for (r, bc, norm, series) in cols }
    pivot5 = pd.DataFrame(data)
    pivot5.index.name = "M"
    pivot5 = pivot5.reindex(M_values)
    pivot5 = pivot5.sort_index(axis=1, level=[0, 1, 2])

    display(pivot5.map(dash_if_nan))


Uniform Mesh : TABLE 4 (Problem 3, R = 1.0)


Simpson's rule                            Trapezoidal rule           \
          Dirichlet           Neumann                 Dirichlet            
             ||·||2   ||·||∞   ||·||2   ||·||∞           ||·||2   ||·||∞   
M                                                                          
64          1.8e-05  2.3e-05  9.5e-06  2.0e-05          1.0e-04  1.1e-04   
128         2.3e-06  3.2e-06  1.2e-06  2.6e-06          2.5e-05  2.9e-05   
256         2.9e-07  6.2e-07  1.5e-07  3.2e-07          6.2e-06  7.5e-06   
512         3.7e-08  1.3e-07  1.8e-08  4.0e-08          1.5e-06  2.0e-06   
1024        4.6e-09  3.0e-08  2.3e-09  5.2e-09          3.9e-07  5.1e-07   
2048        5.9e-10  7.2e-09  3.0e-10  8.7e-10          9.6e-08  1.3e-07   

                        
      Neumann           
       ||·||2   ||·||∞  
M                       
64    3.4e-04  6.3e-04  
128   8.4e-05  1.5e-04  
256   2.1e-05  3.8e-05  
512   5.2e-06  9.5e-06  
1024  1.3e-06  2.4e-06  
2048  3.2e-07  5.9e-07


Fixed Nonuniform Mesh - NUDFT : TABLE 4 (Problem 3, R = 1.0)


Simpson's rule                            Trapezoidal rule           \
          Dirichlet           Neumann                 Dirichlet            
             ||·||2   ||·||∞   ||·||2   ||·||∞           ||·||2   ||·||∞   
M                                                                          
64          1.8e-05  2.3e-05  3.3e-04  3.3e-04          1.0e-04  1.1e-04   
128         2.3e-06  3.2e-06  3.3e-04  3.3e-04          2.5e-05  2.9e-05   
256         2.9e-07  6.2e-07  3.3e-04  3.3e-04          6.2e-06  7.5e-06   
512         3.7e-08  1.3e-07  3.3e-04  3.3e-04          1.5e-06  2.0e-06   
1024        4.6e-09  3.0e-08  3.3e-04  3.3e-04          3.9e-07  5.1e-07   
2048        5.5e-10  7.2e-09  3.3e-04  3.3e-04          9.6e-08  1.3e-07   

                        
      Neumann           
       ||·||2   ||·||∞  
M                       
64    4.8e-04  9.6e-04  
128   3.4e-04  4.9e-04  
256   3.3e-04  3.7e-04  
512   3.3e-04  3.4e-04  
1024  3.3e-04  3.3e-04  
2048  3.3e-04  3.3e-04


Fixed Nonuniform Mesh - NUFFT : TABLE 4 (Problem 3, R = 1.0)


...

Simpson's rule                            Trapezoidal rule           \
          Dirichlet           Neumann                 Dirichlet            
             ||·||2   ||·||∞   ||·||2   ||·||∞           ||·||2   ||·||∞   
M                                                                          
64          4.0e-08  3.7e-06  1.9e-08  5.8e-08          5.4e-06  3.1e-05   
128         4.8e-09  9.0e-07  2.2e-09  7.1e-09          1.3e-06  8.2e-06   
256         5.9e-10  2.2e-07  2.7e-10  8.8e-10          3.3e-07  2.2e-06   
512         7.3e-11  5.5e-08  3.4e-11  1.1e-10          8.2e-08  5.9e-07   
1024        9.0e-12  1.4e-08  4.2e-12  1.4e-11          2.1e-08  1.6e-07   
2048        1.1e-12  3.4e-09  5.2e-13  1.7e-12          5.1e-09  4.2e-08   

                        
      Neumann           
       ||·||2   ||·||∞  
M                       
64    3.6e-06  7.8e-06  
128   8.7e-07  1.9e-06  
256   2.2e-07  4.7e-07  
512   5.4e-08  1.2e-07  
1024  1.3e-08  2.9e-08  
2048  3.4e-09  7.3e-09


Fixed Nonuniform Mesh - NUDFT : TABLE 5 (Problem 3, R = 0.5)


Simpson's rule                            Trapezoidal rule           \
          Dirichlet           Neumann                 Dirichlet            
             ||·||2   ||·||∞   ||·||2   ||·||∞           ||·||2   ||·||∞   
M                                                                          
64          4.0e-08  3.7e-06  4.8e-05  7.8e-05          5.4e-06  3.1e-05   
128         4.8e-09  9.0e-07  4.8e-05  7.8e-05          1.3e-06  8.2e-06   
256         5.9e-10  2.2e-07  4.8e-05  7.8e-05          3.3e-07  2.2e-06   
512         7.3e-11  5.5e-08  4.8e-05  7.8e-05          8.2e-08  5.9e-07   
1024        9.2e-12  1.4e-08  4.8e-05  7.8e-05          2.1e-08  1.6e-07   
2048        1.3e-12  3.4e-09  4.8e-05  7.8e-05          5.1e-09  4.2e-08   

                        
      Neumann           
       ||·||2   ||·||∞  
M                       
64    4.9e-05  8.6e-05  
128   4.8e-05  8.0e-05  
256   4.8e-05  7.9e-05  
512   4.8e-05  7.8e-05  
1024  4.8e-05  7.8e-05  
2048  4.8e-05  7.8e-05


Fixed Nonuniform Mesh - NUFFT : TABLE 5 (Problem 3, R = 0.5)


Simpson's rule                            Trapezoidal rule           \
          Dirichlet           Neumann                 Dirichlet            
             ||·||2   ||·||∞   ||·||2   ||·||∞           ||·||2   ||·||∞   
M                                                                          
64          3.9e-08  3.7e-06  2.6e-05  4.6e-05          5.4e-06  3.1e-05   
128         4.8e-09  9.0e-07  2.6e-05  4.6e-05          1.3e-06  8.2e-06   
256         1.6e-09  2.2e-07  2.6e-05  4.6e-05          3.3e-07  2.2e-06   
512         1.5e-09  5.5e-08  2.6e-05  4.6e-05          8.2e-08  5.9e-07   
1024        1.5e-09  1.4e-08  2.6e-05  4.6e-05          2.1e-08  1.6e-07   
2048        1.5e-09  8.5e-09  2.6e-05  4.6e-05          5.3e-09  4.2e-08   

                        
      Neumann           
       ||·||2   ||·||∞  
M                       
64    2.6e-05  5.3e-05  
128   2.6e-05  4.8e-05  
256   2.6e-05  4.7e-05  
512   2.6e-05  4.6e-05  
1024  2.6e-05  4.6e-05  
2048  2.6e-05  4.6e-05